# Create Pew Biomedical Scholars Awards (FELLOWSHIP PATTERN)

Ingest of The Pew Charitable Trusts' official **Pew Biomedical Scholars** directory at `www.pew.org/en/projects/pew-biomedical-scholars/directory-of-pew-scholars`. The directory page exposes a Sitecore data model with a year-filtered JSON API (`/api/Scholar/GetScholarsByYear`) and per-scholar detail pages. Local source validation on 2026-05-21 found **819 scholars** across **1985-2025**; 816/819 pages publish an institution, and the three source-missing institutions are preserved as NULL rather than inferred from department/title text.

**Awarding body / funder:** Pew Charitable Trusts — `F4320306148`; ROR `https://ror.org/02xhk2825`; DOI `10.13039/100000875`.

**Pattern: FELLOWSHIP.** Each row is one named Pew Biomedical Scholar. `lead_investigator` is person-level: the scholar, with `given_name` and `family_name` parsed from the official detail-page heading. `affiliation.name` maps to the source's detail-page `Institution` field.

**Amount/currency waiver:** the official scholar directory and per-scholar detail pages do not publish historical per-scholar award amounts. Pew's own retrospective material says biomedical program funding has ranged over time, so applying a current award value backward would be an inference. This notebook therefore leaves `amount` and `currency` NULL by source authority.

**Date handling:** the source publishes an award year. Recent official Pew press releases describe the scholars as receiving four years of funding, so the raw script provides a four-year award window (`award_year-01-01` through `award_year+3-12-31`). Admin should inspect the date sanity cells before final approval.

**Source authority:** Pew's own website; method #3 on the runbook ladder (site search/index API discovered from page bundle) plus official detail pages. No Wikipedia/Wikidata backfill.

**Prerequisites:** run `scripts/local/pew_biomedical_scholars_to_s3.py` to fetch the directory model, crawl all advertised years and scholar detail pages, write `pew_biomedical_scholars.parquet`, and upload to S3. Contractor handoff note: local validation can run with `--skip-upload`; admin must upload/run in Databricks because contractor has no S3/Databricks credentials.

**S3 location:** `s3a://openalex-ingest/awards/pew_biomedical_scholars/pew_biomedical_scholars.parquet`


## Step 1: Create staging table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.pew_biomedical_scholars_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/pew_biomedical_scholars/pew_biomedical_scholars.parquet`;

In [ ]:
%sql
SELECT COUNT(*) AS total_raw_rows FROM openalex.awards.pew_biomedical_scholars_raw;

## Step 1.5: Inspect raw data first

Expected source columns include `funder_award_id`, `slug`, `full_name`, `name_without_credentials`, `given_name`, `family_name`, `award_year`, `start_date`, `end_date`, `display_name`, `description`, `research_text`, `research_fields`, `research_fields_text`, `keywords`, `keywords_text`, `job_title`, `department`, `institution`, `address`, `city_state_zip`, `email`, `website`, `profile_url`, `item_url`, `api_image_url`, `detail_image_url`, `image_alt`, `api_full_name`, `api_job_title`, `api_institution`, `source_api_root_id`, `source_api_year`, `source_directory_url`, `amount`, `currency`, `amount_note`, and `downloaded_at`.

Money scan note: the raw file intentionally carries NULL `amount`/`currency` for OpenAlex mapping. The source directory has no per-scholar amount field; do not infer one from current program announcements without a separate historical amount audit.


In [ ]:
%sql
DESCRIBE openalex.awards.pew_biomedical_scholars_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.pew_biomedical_scholars_raw LIMIT 10;

In [ ]:
%sql
SELECT
    COUNT(amount) AS rows_with_amount,
    COUNT(currency) AS rows_with_currency,
    COUNT(amount_note) AS rows_with_amount_note,
    COUNT(*) AS total_rows
FROM openalex.awards.pew_biomedical_scholars_raw;

In [ ]:
%sql
SELECT
    MIN(TRY_CAST(award_year AS INT)) AS min_award_year,
    MAX(TRY_CAST(award_year AS INT)) AS max_award_year,
    COUNT(DISTINCT award_year) AS distinct_award_years,
    COUNT(DISTINCT funder_award_id) AS distinct_award_ids,
    COUNT(*) AS total_rows
FROM openalex.awards.pew_biomedical_scholars_raw;

## Step 1.6: Funder existence fail-fast

This must return exactly one row before the transform runs. If it returns zero, stop and ask admin to resolve the funder table before inserting awards.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320306148;

## Step 2: Transform to OpenAlex award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.pew_biomedical_scholars_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306148
)
SELECT
    abs(xxhash64(CONCAT(
        TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
    ))) % 9000000000 AS id,
    n.display_name,
    n.description,
    f.funder_id,
    n.funder_award_id,
    CAST(NULL AS DOUBLE) AS amount,
    CAST(NULL AS STRING) AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'fellowship' AS funding_type,
    'Pew Biomedical Scholars' AS funder_scheme,
    'pew_biomedical_scholars_directory' AS provenance,
    TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(n.end_date, 'yyyy-MM-dd') AS end_date,
    TRY_CAST(n.award_year AS INT) AS start_year,
    TRY_CAST(n.award_year AS INT) + 3 AS end_year,
    CASE
        WHEN n.full_name IS NULL OR n.full_name = '' THEN NULL
        ELSE struct(
            n.given_name AS given_name,
            n.family_name AS family_name,
            CAST(NULL AS STRING) AS orcid,
            TRY_TO_DATE(n.start_date, 'yyyy-MM-dd') AS role_start,
            struct(
                n.institution AS name,
                CAST(NULL AS STRING) AS country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
            ) AS affiliation
        )
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    n.profile_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT(
               TRY_CAST(f.funder_id AS STRING), ':', LOWER(n.funder_award_id)
           ))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM openalex.awards.pew_biomedical_scholars_raw n
CROSS JOIN funder_resolved f
WHERE n.funder_award_id IS NOT NULL
  AND n.full_name IS NOT NULL
  AND n.award_year IS NOT NULL;

## Step 2.5: Validate transformed awards

In [ ]:
%sql
SELECT
    COUNT(*) AS awards,
    COUNT(DISTINCT funder_award_id) AS distinct_award_ids,
    COUNT(display_name) AS with_display_name,
    COUNT(description) AS with_description,
    COUNT(lead_investigator.given_name) AS with_given_name,
    COUNT(lead_investigator.family_name) AS with_family_name,
    COUNT(lead_investigator.affiliation.name) AS with_affiliation,
    COUNT(amount) AS with_amount,
    MIN(start_year) AS min_start_year,
    MAX(start_year) AS max_start_year
FROM openalex.awards.pew_biomedical_scholars_awards;

In [ ]:
%sql
SELECT funder_award_id, COUNT(*) AS n
FROM openalex.awards.pew_biomedical_scholars_awards
GROUP BY funder_award_id
HAVING COUNT(*) > 1;

In [ ]:
%sql
SELECT
    start_year,
    COUNT(*) AS scholars,
    COUNT(description) AS with_research_text,
    COUNT(lead_investigator.affiliation.name) AS with_institution
FROM openalex.awards.pew_biomedical_scholars_awards
GROUP BY start_year
ORDER BY start_year;

In [ ]:
%sql
SELECT
    display_name,
    funder_award_id,
    start_year,
    lead_investigator,
    LEFT(description, 500) AS description_sample,
    landing_page_url
FROM openalex.awards.pew_biomedical_scholars_awards
ORDER BY start_year DESC, display_name
LIMIT 20;

## Step 3: Insert into `openalex_awards_raw` with priority 97

Priority 97 is reserved for this Pew Biomedical Scholars source. Priorities 66-96 are reserved by in-flight PR backlog items, including Claude's priority 94, Packard priority 95, and Stockholm Water Prize priority 96.


In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pew_biomedical_scholars_directory' AND priority = 97;

-- Insert into openalex_awards_raw with priority
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    97 AS priority
FROM openalex.awards.pew_biomedical_scholars_awards;

## Step 4: Post-insert QA

In [ ]:
%sql
SELECT
    provenance,
    priority,
    COUNT(*) AS rows,
    COUNT(DISTINCT funder_award_id) AS distinct_award_ids,
    COUNT(amount) AS rows_with_amount,
    MIN(start_year) AS min_year,
    MAX(start_year) AS max_year
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pew_biomedical_scholars_directory'
  AND priority = 97
GROUP BY provenance, priority;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, start_year, lead_investigator, landing_page_url
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'pew_biomedical_scholars_directory'
  AND priority = 97
ORDER BY start_year DESC, display_name
LIMIT 20;